In [7]:
# Import necessary libraries
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("./data/cybersecurity_attacks.csv")

# Hypothesis 1: Chi-square Test with Improved Feature Engineering
attack_mapping = {
    "DDoS": "Network Attack", "Phishing": "Social Engineering", "SQL Injection": "Web Attack",
    "Brute Force": "Authentication Attack", "Ransomware": "Malware Attack", "Trojan": "Malware Attack",
    "Spyware": "Malware Attack", "Adware": "Malware Attack", "Rootkit": "Malware Attack", "Worm": "Malware Attack",
    "Backdoor": "Malware Attack", "XSS": "Web Attack", "Man-in-the-Middle": "Network Attack",
    "Privilege Escalation": "System Attack", "Zero-Day Exploit": "System Attack"
}
df["General Attack Type"] = df["Attack Type"].map(lambda x: attack_mapping.get(x, "Other"))
df["Anomaly Category"] = pd.cut(df["Anomaly Scores"], bins=[0, 0.3, 0.7, 1], labels=["Low", "Medium", "High"])
df["Malware Indicators"].fillna("None", inplace=True)
contingency_table = pd.crosstab(df["General Attack Type"], df["Malware Indicators"])
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

print("Hypothesis 1: Chi-square Test with Improved Feature Engineering")
print("Chi-square Statistic:", chi2_stat)
print("P-value:", p_value)
print("Degrees of Freedom:", dof)
print("Expected Frequencies:", expected)
print()

Hypothesis 1: Chi-square Test with Improved Feature Engineering
Chi-square Statistic: 1.587516828620064
P-value: 0.20768126877193643
Degrees of Freedom: 1
Expected Frequencies: [[ 6714.  6714.]
 [13286. 13286.]]



In [ ]:
# Hypothesis 2: Machine Learning Model Improvements
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Hour"] = df["Timestamp"].dt.hour
df["Day of Week"] = df["Timestamp"].dt.dayofweek
ml_features = ["Protocol", "Packet Length", "Anomaly Scores", "Severity Level", "Traffic Type",
               "Attack Signature", "Firewall Logs", "IDS/IPS Alerts", "Proxy Information",
               "Action Taken", "Hour", "Day of Week"]
df_ml = df.dropna(subset=ml_features + ["Attack Type"])

label_encoders = {}
for col in ml_features:
    if df_ml[col].dtype == "object":
        le = LabelEncoder()
        df_ml[col] = le.fit_transform(df_ml[col])
        label_encoders[col] = le

target_encoder = LabelEncoder()
df_ml["Attack Type"] = target_encoder.fit_transform(df_ml["Attack Type"])
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(df_ml[ml_features], df_ml["Attack Type"], test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_ml[["Packet Length", "Anomaly Scores", "Hour", "Day of Week"]] = scaler.fit_transform(X_train_ml[["Packet Length", "Anomaly Scores", "Hour", "Day of Week"]])
X_test_ml[["Packet Length", "Anomaly Scores", "Hour", "Day of Week"]] = scaler.transform(X_test_ml[["Packet Length", "Anomaly Scores", "Hour", "Day of Week"]])

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
stacking_model = StackingClassifier(
    estimators=[("rf", rf_model), ("gb", gb_model), ("mlp", mlp_model)],
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=42)
)
stacking_model.fit(X_train_ml, y_train_ml)
y_pred_ml = stacking_model.predict(X_test_ml)
accuracy_ml = accuracy_score(y_test_ml, y_pred_ml)
precision_ml = precision_score(y_test_ml, y_pred_ml, average="weighted")
recall_ml = recall_score(y_test_ml, y_pred_ml, average="weighted")
f1_ml = f1_score(y_test_ml, y_pred_ml, average="weighted")


In [9]:
# Display results
print("Hypothesis 2: Machine Learning Model Improvements")
print("Accuracy:", accuracy_ml)
print("Precision:", precision_ml)
print("Recall:", recall_ml)
print("F1 Score:", f1_ml)
print()

Hypothesis 2: Machine Learning Model Improvements
Accuracy: 0.3562874251497006
Precision: 0.3598696523087062
Recall: 0.3562874251497006
F1 Score: 0.3546470845008658

